Program structure: 
isValid = True, errormessage = "", if ever failed set to false and add text. 
1. BIDS Validator
2. Load with muniverse, helpful for finding files (that could be inherited etc). should work if validator passed. 
3. loop over everything, apply checks 
-> write to scores.json. valid = 1 if valid. valid = 0 and errors="..." if not. 

In [ ]:
import json

localpath = "C:/Users/nikla/Documents/HiwiStuttgart/munitquest-data-submission"

# app/input is where folder lives.

import subprocess

import muniverse.utils.bids_routines
import pandas as pd 

# Bids validator & bids validator config

In [ ]:
# creating a validation json and then writing it into a file, because this way i can put comments here, which directly editing the .json file would not allow.

validationDict = { # JSON configuration file that reclassifies issues as warnings, errors or ignored.
    "ignore": [],
    "warning": [
        {
            "code": "SIDECAR_WITHOUT_DATAFILE",  # ignore missing edfs
        }, 
    ],
    "error": [],
}

with open(localpath + "/bundle/scoring/bidsValidatorConfig.json", "w") as f:
    json.dump(validationDict, f, indent=4)

In [ ]:
def run_bids_validator_with_config(
    path,
    ignored_codes=[],
    ignored_fields=[],
    ignored_files=[],
    print_errors=True,
    print_warnings=True,
):
    """
    API to the official BIDS validator.

    Args:
        path (str): Absolute of relative path to your BIDS dataset
        ignored_codes (list of str): Ignored error codes (e.g. ["SIDECAR_KEY_RECOMMENDED"])
        ignored_fileds (list of str): Errors corresponding to that field are ignored (e.g. ["DeviceSerialNumber"])
        ignored_files (list of str): Ignored errors in these files (e.g. ["/dataset_description.json"])
        print_errors (bool): Descides if errors should be printed
        print_warnings (bool): Descides if warnings should be printed

    Returns:
        errors (list): List of detected errors
        warnings (list): List of detected warnings
        valid (bool): True if there are no errors

    """

    # Run bids validator
    result = subprocess.run(
        [
            "bids-validator-deno",
            "--format",
            "json",
            "--config",
            localpath + "/bundle/scoring/bidsValidatorConfig.json",  # specifies config file to use
            path,
        ],
        capture_output=True,
        text=True,
    )
    # Extract and filter all issues
    validation = json.loads(result.stdout)
    issues = validation["issues"]["issues"]
    issues = [f for f in issues if (not "code" in f or f["code"] not in ignored_codes)]
    issues = [
        f for f in issues if (not "subCode" in f or f["subCode"] not in ignored_fields)
    ]
    issues = [
        f for f in issues if (not "location" in f or f["location"] not in ignored_files)
    ]
    # Split issues in errors and warnings
    errors = [f for f in issues if f["severity"] == "error"]
    warnings = [f for f in issues if f["severity"] == "warning"]
    # Print issues
    if print_errors:
        print(f"Number of detected errors: {len(errors)}")
        print(json.dumps(errors, indent=4))
    if print_warnings:
        print(f"Number of detected warnings: {len(warnings)}")
        print(json.dumps(warnings, indent=4))
    # Check if the folder is BIDS valid
    valid = True if len(errors) == 0 else False

    return errors, warnings, valid


    result = subprocess.run(
        [
            "bids-validator-deno",
            "--format",
            "json",
            "--config",
            localpath + "/bundle/scoring/config.json",  # specifies config file to use
            path,
        ],
        capture_output=True,
        text=True,
    )
    return result

In [ ]:
# get rid of muniverse warnings about inheritance 
import warnings
warnings.simplefilter("ignore")

# Checking functions dataset wide

In [ ]:
def checkDatasetDescription(dataset:muniverse.utils.bids_routines.bids_dataset):
    passed = True 
    error = "" 
    if not "EthicsApprovals" in dataset.dataset_sidecar.keys(): # TODO also check for non-emptiness? currently an empty list passes test. 
        passed=False 
        error+="'EthicsApprovals' not present in 'dataset_description.json'."
    return passed, error 

# Checking functions recording specific


In [ ]:
def isRecordingABaselineNoiseRecording(recording:muniverse.utils.bids_routines.bids_emg_recording): 
    """Return True if task_label follows convention for naming restnoise/baselinenoise recordings.""" # TODO ask thomas what this convention should be 
    if recording.task_label == "baselinenoise": 
        return True 
    else: 
        return False 

def checkElectrodesTSV(recording:muniverse.utils.bids_routines.bids_emg_recording):
    passed = True 
    error = "" 
    if len(recording.electrodes) == 0: # No need for try-except because muniverse creates an empty dataframe even if electrodes.tsv doesn't exist 
        passed = False 
        error = "electrodes.tsv missing or empty" 
    return passed, error 


def checkEventsTSV(recording:muniverse.utils.bids_routines.bids_emg_recording):
    passed = True 
    error = "" 
    if not isRecordingABaselineNoiseRecording(recording): # baselinenoise tasks are allowed to have no events 
        if len(recording.events) == 0: # no need for try-except because even if events.tsv doesn't exist, muniverse creates empty dataframe 
            passed = False 
            error = "events.tsv missing or empty" 
        else: 
            try: 
                assert ("muscle_on" in recording.events["event_type"].values) and ("muscle_on" in recording.events["event_type"].values) # It's important to wrap this into try-except because script would crash if we try to access dataframe column that does not exist. 
            except: 
                passed = False 
                error = "events.tsv does not contain 'muscle_on' and 'muscle_off' in column 'event_type'. "

    return passed, error 


def checkEmgJson(recording:muniverse.utils.bids_routines.bids_emg_recording): 
    passed = True 
    error = "" 
    try: 
        assert recording.emg_sidecar["SamplingFrequency"] >= 2000
    except: 
        passed = False 
        error += "'SamplingFrequency' in _emg.json not present or not >= 2000."

    if not "Gain" in recording.emg_sidecar.keys(): 
        passed = False 
        error += "'Gain' in _emg.json not present."
        
    if not "Preamplification" in recording.emg_sidecar.keys(): 
        passed = False 
        error += "'Preamplification' in _emg.json not present."
    return passed, error 

# Checking

In [ ]:
testTheseDatasets = ["Avrillon", "Caillet", "CailletSomeEventsTSVMissing", "CailletSomeCoordJsonMissing", "CailletSomeElectrodeTSVMissing", "CailletSomeMuscleOnEventMissing", "CailletSomeEmgJsonMissing", "CailletNoEthicsApproval"]
# testTheseDatasets = ["CailletSomeElectrodeTSVAndCoordJsonMissing"] # currently crashes 

for datasetName in testTheseDatasets: 
    print(f"\n \nstarting to check dataset: {datasetName}")
    datasetPassesTest = True 
    datasetTestErrors = ""
    errors, warnings, valid = run_bids_validator_with_config(
        localpath + "/tests/" + datasetName,
        print_warnings=False, 
        print_errors=False, 
    )
    if len(errors) > 0: 
        datasetPassesTest = False 
        datasetTestErrors += "BIDS validator failed with the following errors: \n" + str(errors)
    
    if datasetPassesTest: # only load it via muniverse if it passed validator, otherwise probably some horrible errors
        # load entire dataset 
        dataset = muniverse.utils.bids_routines.bids_dataset(
                datasetname=datasetName, 
                root=localpath + "/tests/", 
                overwrite=False
        )
        dataset.read()

        # ALL THE CHECKS WE WANT TO PERFORM (dataset wide)
        listOfCheckingFunctionsDatasetWide = [checkDatasetDescription]
        for checkingFunction in listOfCheckingFunctionsDatasetWide: 
            p, e = checkingFunction(dataset) 
            if not p: 
                datasetPassesTest = False 
                datasetTestErrors += f"\n {e}"


        
        # Create dataframe that lists all "_emg.json" files in the dataset folder, so that we can use it to loop over every individual recording. (Obviously this method would not catch a missing "_emg.json" file (bids-validator doesn't care either))
        allfiles = dataset.list_all_files(suffix="emg", extension="json") # "_emg.json" cannot be inherited, so should always exist for each recording 
        df = allfiles.copy(deep=True)
        keepTheseColumns = ["sub", "ses", "task", "acq", "run", "recording"]
        dropTheseColumns = [x for x in df.columns if not x in keepTheseColumns]
        df.drop(columns=dropTheseColumns, inplace=True)


        for i in df.index: # Loop over every recording for which an "_emg.json" exists 
            sub = df.loc[i,"sub"]
            ses = df.loc[i,"ses"]
            if type(ses) == pd.api.typing.NAType: ses = None 
            task = df.loc[i,"task"]
            acq = df.loc[i,"acq"]
            if type(acq) == pd.api.typing.NAType: acq = None 
            run = df.loc[i,"run"]
            recording = df.loc[i,"recording"]
            if type(recording) == pd.api.typing.NAType: recording = None 

            currentRecordingString = f"sub: {sub}, session: {ses}, task: {task}, acq: {acq}, run: {run}, recording:{recording}"

            recording = muniverse.utils.bids_routines.bids_emg_recording(
                parent_dataset=dataset, 
                subject_label=sub,
                session_label=ses,
                task_label=task,
                acq_label=acq,
                run_label=run,
                recording_label=recording,
                # inherited_metadata=["coordsystem.json", "electrodes.tsv","events.json"], 
                # inherited_level=["subject", "subject", "dataset"], 
            )
            recording.read()

            listOfCheckingFunctionsRecordingSpecific = [checkElectrodesTSV, checkEventsTSV, checkEmgJson]

            # ALL THE CHECKS WE WANT TO PERFORM (Recording specific)
            for checkingFunction in listOfCheckingFunctionsRecordingSpecific: 
                p, e = checkingFunction(recording) 
                if not p: 
                    datasetPassesTest = False 
                    datasetTestErrors += f"\n   Error in {currentRecordingString}: \n {e}"


    print(f"dataset passes? {datasetPassesTest}")
    print(f"errors: {datasetTestErrors}")


In [ ]:
# placeholder: dict[str, float] = {"valid": 1}  # 1 if passed, 0 if not.

# with open(localpath + "/app/output/scores.json", "w") as f:
#     json.dump(placeholder, f, indent=4)